# MNIST GB-RBM: TSI vs DSI vs CVSI

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from mnist_ebm import (GaussianBernoulliRBM, LogProbEnergy, Standardizer,
                       load_mnist, train_rbm, make_rbm_gibbs_sampler)
from estimators import score_from_samples, make_estimator
from dem.models.components.cvsi import get_a2_b2
from dem.models.components.sdes import ReverseSDE          # VE- *and* VP-general
from dem.models.components.sde_integration import integrate_sde
from vp_utils import build_schedule

torch.set_default_dtype(torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"   # auto-detect GPU
print("device:", device, "| torch", torch.__version__)

## 1. Configuration

In [ ]:
# --- data / energy model ---
RES, N_TRAIN, D = 14, 50000, 14 * 14
N_HIDDEN, RBM_EPOCHS = 124, 200
RBM_PATH = f"models/rbm_gb_mnist{14}.pt"

# --- noise schedule: "VE" or "VP" -------------------------------------------------
# "VE": variance exploding (a(t)=1, b^2=h(t)), the original iDEM setting.
# "VP": SNR/TV variance preserving (a(t)->0, total variance held at 1). Since
#       TSI = grad log p / a(t), VP amplifies the energy-gradient branch at high noise.
#       VP assumes unit-variance data, which `Standardizer` already provides, so no
#       coordinate rescaling is needed and Z-space is the sampling space under both.
#       The standardization is a single scalar, so per-pixel variances are not equal;
#       constant TV preserves a^2 Var + b^2 per direction only for isotropic data.
SCHEDULE = "VE"

# --- reverse-SDE start time (see vp_utils.build_schedule) -------------------------
# None = the schedule's own high-noise end (VE: 1.0, VP: t_max=0.99). Under VP a(t)->0
# near t_max makes the tilted target degenerate; ~0.9 skips that sliver at nearly no
# cost, since constant TV=1 keeps the marginal there close to the prior.
GEN_START_T = None if SCHEDULE == "VE" else 0.9

# --- variance-exploding noise schedule (VE only; VP's SNR/TV schedule is fixed) ---
SIGMA_MIN, SIGMA_MAX = 5e-3, 50.0

# --- exact posterior sampler (the ONLY sampler used in this notebook) ---
GIBBS_STEPS = 200      # blocked-Gibbs sweeps (cheap matmuls, no autograd)

# --- estimator variance / error sweep ---
T_GRID   = 12          # noise levels
N_XT     = 8           # distinct x_t per noise level
N_REPEAT = 25          # independent re-estimates -> variance across repeats
K_WORK   = 64          # MC samples per working estimate
K_REF    = 2048        # Gibbs budget for the (near-truth) reference score
N_REF_REP = 8          # reference averaged over this many large-budget draws

# --- reverse-SDE generation (end-to-end, all with the Gibbs sampler) ---
N_GEN, T_STEPS, K_SAMPLE = 36, 64, 8
print(f"SCHEDULE = {SCHEDULE}  |  D = {D}")


In [ ]:
import os
X, y = load_mnist(resolution=RES, n_max=N_TRAIN)
std = Standardizer(X)
Z = std.forward(X).to(device)                     # unit-variance space: diffusion lives here

rbm = GaussianBernoulliRBM(D, n_hidden=N_HIDDEN, sigma=1.0).to(device)
if os.path.exists(RBM_PATH):
    rbm.load_state_dict(torch.load(RBM_PATH, weights_only=True, map_location=device))
    print(f"loaded cached RBM from {RBM_PATH}")
else:
    train_rbm(rbm, Z, epochs=RBM_EPOCHS, batch_size=256, lr=1e-4,
              persistent=True, sample_neg=True, seed=42)
    os.makedirs("models", exist_ok=True)
    torch.save(rbm.state_dict(), RBM_PATH)
    print(f"saved RBM to {RBM_PATH}")

energy = LogProbEnergy(rbm)

# --- schedule (VE or VP; see the SCHEDULE flag above) ------------------------------
# Z is already unit-variance, so no coordinate rescaling is needed under VP (scale=1.0)
# and Z-space is the sampling space under both schedules.
sched = build_schedule(SCHEDULE, sigma_min=SIGMA_MIN, sigma_max=SIGMA_MAX,
                       scale=1.0, gen_start=GEN_START_T)
noise = sched.noise

# Noise-level grid, over the schedule's own active range (VE: the original
# 0.08 -> 1.0; VP: t_min -> gen_start).
T_LO = 0.08 if SCHEDULE == "VE" else float(noise.t_min)
T_HI = float(sched.gen_start)

def ab_of(t):
    """(a^2, b^2) of the kernel N(a x0, b^2 I) at time t. Under VE: (1, h(t))."""
    return get_a2_b2(torch.as_tensor(t, device=device), noise)

print(sched.summary())
print(f"Z per-dim std (global) = {Z.std().item():.3f}  (VP wants ~1)   "
      f"| t-grid [{T_LO:.4g}, {T_HI:.4g}]")

### 1a. Reference: sampling directly from the trained RBM

In [ ]:
def show_grid(imgs, title):
    imgs = std.inverse(rbm.denoise(imgs)).reshape(-1, RES, RES).clamp(0, 1).cpu()
    n = int(np.ceil(np.sqrt(len(imgs))))
    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for ax, im in zip(axes.flat, imgs):
        ax.imshow(im, cmap="gray"); ax.axis("off")
    for ax in axes.flat[len(imgs):]:
        ax.axis("off")
    fig.suptitle(title, y=1.0); plt.tight_layout(); plt.show()

def nn_labels(imgs):
    d = torch.cdist(rbm.denoise(imgs), Z)
    return y[d.argmin(dim=1).cpu()]

torch.manual_seed(42)
v_init = torch.randn(N_GEN, D, device=device)
v_model = rbm.sample(v_init, n_steps=2000, sample_v=True)

with torch.no_grad():
    labs_model = nn_labels(v_model)
    F_model = rbm.free_energy(v_model).mean().item()
    F_data = rbm.free_energy(Z[:2000]).mean().item()

print(f"long Gibbs on p(x):  mean F = {F_model:8.1f}  unique digits = {len(labs_model.unique())}/10  "
      f"counts={torch.bincount(labs_model, minlength=10).tolist()}")
print(f"data (reference):    mean F = {F_data:8.1f}")

show_grid(v_model, "reference: long Gibbs chain on p(x) directly")

## 2. One sampler, three estimators

In [ ]:
gibbs_sampler = make_rbm_gibbs_sampler(rbm, n_steps=GIBBS_STEPS)

ESTS   = ["tsi", "dsi", "cvsi"]
from style import PAPER_COLORS as COLORS
LABEL  = {"tsi": "TSI", "dsi": "DSI", "cvsi": "CVSI"}


def estimators_on_gibbs(t, xt, K, ests=ESTS, return_ct=False):
    '''Draw K exact-Gibbs posterior samples at (t, xt) and evaluate each
    estimator on the SAME samples. Returns {est: score (B,D)} (+ c~*(t) if asked).'''
    x0, w, h_t, _ = gibbs_sampler(t, xt, noise, energy, K)
    out = {}
    ct = None
    for e in ests:
        if e == "cvsi" and return_ct:
            out[e], ct = score_from_samples(t, xt, x0, w, h_t, energy, noise,
                                            estimator=e, return_ct=True)
        else:
            out[e] = score_from_samples(t, xt, x0, w, h_t, energy, noise, estimator=e)
    return (out, ct) if return_ct else out

## 3. Estimator variance across noise levels

In [ ]:
t_grid = torch.linspace(T_LO, T_HI, T_GRID, device=device)
torch.manual_seed(0)
idx = torch.randperm(len(Z), device=device)[:N_XT]
x0_fix = Z[idx]

var_curves = {e: [] for e in ESTS}
for t in tqdm(t_grid, desc="Section 3: variance sweep"):
    tt = t.repeat(N_XT)
    # forward kernel N(a(t) x0, b^2(t) I); under VE this is x0 + sqrt(h) * randn exactly
    a2, b2 = ab_of(t)
    xt = a2.sqrt() * x0_fix + b2.sqrt() * torch.randn_like(x0_fix)  # (N_XT, D), fixed across repeats
    # (N_REPEAT, N_XT, D) stacks of each estimator
    stacks = {e: [] for e in ESTS}
    for r in range(N_REPEAT):
        out = estimators_on_gibbs(tt, xt, K_WORK)
        for e in ESTS:
            stacks[e].append(out[e])
    for e in ESTS:
        s = torch.stack(stacks[e], dim=0)                        # (N_REPEAT, N_XT, D)
        v = s.var(dim=0, unbiased=True).sum(dim=-1)              # trace-cov per x_t -> (N_XT,)
        var_curves[e].append(v.mean().item())                   # average over x_t

t_np = t_grid.cpu().numpy()
plt.figure(figsize=(6.4, 4.0))
for e in ESTS:
    plt.plot(t_np, var_curves[e], "o-", color=COLORS[e], label=LABEL[e], ms=5)
plt.yscale("log"); plt.xlabel("t"); plt.ylabel(r"estimator variance  $E\,\|\hat s - E\hat s\|^2$")
plt.title(f"Score-estimator variance under exact Gibbs samples (K={K_WORK}, {SCHEDULE})")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

rows = [f"{'t':>6s}" + "".join(f"{LABEL[e]:>12s}" for e in ESTS)]
rows += [f"{t:6.2f}" + "".join(f"{var_curves[e][i]:12.3g}" for e in ESTS)
         for i, t in enumerate(t_np)]
print("\n".join(rows))

## 4. End-to-end reverse-SDE generation per estimator

In [ ]:
def make_score_fn(estimator, K):
    est = make_estimator(sampler=gibbs_sampler, estimator=estimator)
    def score(t, x):
        return est(t, x, energy, noise, num_mc_samples=K)
    return score

def mc_sample(estimator, K, seed=42):
    # ReverseSDE = g^2*score - f_fwd(x,t); f_fwd is 0 under VE, so this reproduces the
    # old VEReverseSDE path exactly, and adds VP's forward contraction when present.
    torch.manual_seed(seed)
    sde = ReverseSDE(make_score_fn(estimator, K), noise)
    x_init = torch.randn(N_GEN, D, device=device) * sched.prior_std
    traj = integrate_sde(sde, x_init, num_integration_steps=T_STEPS,
                         energy_function=energy, diffusion_scale=1.0, no_grad=True,
                         negative_time=False, num_negative_time_steps=1,
                         start_time=sched.gen_start, end_time=sched.gen_end)
    return traj[-1].detach()

samples = {}
for e in tqdm(ESTS, desc="Section 4: reverse-SDE generation"):
    samples[e] = mc_sample(e, K_SAMPLE, seed=42)

In [ ]:
# show_grid, nn_labels defined in section 1a (reused here)
with torch.no_grad():
    rows = [f"{'estimator':12s} {'unique digits':>14s} {'mean F':>10s}"]
    for e in ESTS:
        labs = nn_labels(samples[e])
        F_mean = rbm.free_energy(samples[e]).mean().item()
        rows.append(f"{LABEL[e]:12s} {len(labs.unique()):>10d}/10 {F_mean:>10.1f}"
                    f"   counts={torch.bincount(labs, minlength=10).tolist()}")
    F_data = rbm.free_energy(Z[:2000]).mean().item()
    rows.append(f"{'data (ref)':12s} {'-':>13s} {F_data:>10.1f}")
    rows.append(f"{'model (ref)':12s} {'-':>13s} {F_model:>10.1f}   (section 1a, long Gibbs on p(x))")
print("\n".join(rows))

for e in ESTS:
    show_grid(samples[e], f"reverse-SDE, exact-Gibbs posterior, estimator: {LABEL[e]}")

## 5. Sample-quality metrics

In [ ]:
import mnist_metrics as M

# ---- config for the metric evaluation (bump N_EVAL / N_REF for tighter estimates) ----
N_EVAL          = 5000      # generated samples per estimator (the 36 above are too few for MMD/PR)
N_REF           = 5000     # reference-set size, for both the data and the Gibbs reference
GIBBS_REF_STEPS = 2000     # burn-in for the exact-Gibbs p(x) reference chains
CLF_PATH        = f"models/mnist_cnn{RES}.pt"

classifier = M.load_or_train_classifier(
    CLF_PATH, X, y, side=RES, n_classes=10, device=device, epochs=5)

# Same clean-image readout as show_grid, for any RBM state (generated or Gibbs).
def readout_state(v):
    with torch.no_grad():
        return std.inverse(rbm.denoise(v)).clamp(0, 1)

# Two references, prepared with the metric-agnostic image API:
#   - REAL DATA: true MNIST images (how "MNIST-like" are the samples?)
#   - Asymptotically exact GIBBS p(x): long-Gibbs draws from the RBM (how faithful to what the RBM models?)
torch.manual_seed(123)
ref_data  = std.inverse(Z[torch.randperm(len(Z), device=device)[:N_REF]]).clamp(0, 1)
ref_gibbs = readout_state(rbm.sample(torch.randn(N_REF, D, device=device),
                                     n_steps=GIBBS_REF_STEPS, sample_v=True))
print(f"references ready: data={tuple(ref_data.shape)}  gibbs={tuple(ref_gibbs.shape)}")


In [ ]:
def generate(estimator, n, K=K_SAMPLE, steps=T_STEPS, seed=42):
    """Reverse-SDE generation (as Section 4) but with a configurable batch size,
    returning the clean image readout used everywhere else."""
    torch.manual_seed(seed)
    sde = ReverseSDE(make_score_fn(estimator, K), noise)
    x_init = torch.randn(n, D, device=device) * sched.prior_std
    traj = integrate_sde(sde, x_init, num_integration_steps=steps,
                         energy_function=energy, diffusion_scale=1.0, no_grad=True,
                         negative_time=False, num_negative_time_steps=1,
                         start_time=sched.gen_start, end_time=sched.gen_end)
    return readout_state(traj[-1].detach())

gen_imgs = {}
for e in tqdm(ESTS, desc="Section 5: generating eval batch"):
    gen_imgs[e] = generate(e, N_EVAL)

results_by_ref = {}
for ref_name, ref in [("REAL DATA", ref_data), ("EXACT GIBBS p(x)", ref_gibbs)]:
    results = {LABEL[e]: M.evaluate_samples(gen_imgs[e], ref, classifier=classifier)
               for e in ESTS}
    half = ref.shape[0] // 2
    results["ref (floor)"] = M.evaluate_samples(ref[:half], ref[half:], classifier=classifier)
    results_by_ref[ref_name] = results
    print(f"\n### Sample-quality metrics vs {ref_name}  ({SCHEDULE})")
    print(M.format_table(results))

## 6. Sample quality vs the number of posterior samples $K$

In [ ]:
KEY_METRICS = {"feat_fid": "lower", "class_tv": "lower"}

K_SWEEP = [2, 4, 8, 16, 32]     # MC budget per score estimate
N_EVAL_SWEEP = 5000                  # generated samples per (estimator, K); lower for a quick/CPU run
                                     # runtime scales with K * N_EVAL_SWEEP; T_STEPS stays fixed

half = ref_gibbs.shape[0] // 2
floor_ref = M.evaluate_samples(ref_gibbs[:half], ref_gibbs[half:], classifier=classifier)

sweep = {e: {m: [] for m in KEY_METRICS} for e in ESTS}
for K in tqdm(K_SWEEP, desc="Section 6: K sweep"):
    for e in ESTS:
        imgs = generate(e, N_EVAL_SWEEP, K=K, steps=T_STEPS, seed=42)
        res = M.evaluate_samples(imgs, ref_gibbs, classifier=classifier)
        for m in KEY_METRICS:
            sweep[e][m].append(res[m])

fig, axes = plt.subplots(1, len(KEY_METRICS), figsize=(3.6 * len(KEY_METRICS), 3.8), sharex=True)
for ax, (m, direction) in zip(axes, KEY_METRICS.items()):
    for e in ESTS:
        ax.plot(K_SWEEP, sweep[e][m], "o-", color=COLORS[e], label=LABEL[e], ms=5)
    ax.axhline(floor_ref[m], color="gray", ls="--", lw=1, label="ref (floor)")
    ax.set_xscale("log", base=2)
    ax.set_xticks(K_SWEEP); ax.set_xticklabels(K_SWEEP)
    ax.set_xlabel("K (posterior samples per score estimate)")
    ax.set_title(f"{m}  ({direction} is better)", fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("metric value")
axes[0].legend(fontsize=8)
plt.suptitle(f"Sample quality vs K  (T_STEPS={T_STEPS} fixed), exact-Gibbs $p(x)$ reference",
             y=1.04)
plt.tight_layout(); plt.show()

for m in KEY_METRICS:
    print(f"\n{m}:")
    print(f"{'K':>4s}" + "".join(f"{LABEL[e]:>12s}" for e in ESTS) + f"{'floor':>12s}")
    for i, K in enumerate(K_SWEEP):
        print(f"{K:4d}" + "".join(f"{sweep[e][m][i]:12.4g}" for e in ESTS)
              + f"{floor_ref[m]:12.4g}")
